# Resonator sweep interactive fitter 
This notebook provides a template for interactive resonator fitting for sweeps over a parameter (temperature, blackbody temperature, microwave power, etc). The parameters the user must setup are as follows:
| **Parameter** | Description         |
|--------------|--------------------------------|
| **get_res_data**    | Function to load the data. See template below. |
| **resfit_param**    | name of the resonator fit parameter to plot (e.g. 'fr', 'a').     |
| **nres**    | Number of resonators in the dataset.    |
| **x_name**    | Name of the sweep parameter for the plot (e.g. 'Temperature', 'BB temperature', 'Puw').    |
| **x_unit**    | Unit of the sweep parameter data for the plot (e.g. 'mK', 'K', 'W').    |
| **x_df_name**    | Name of the sweep parameter for inserting into the output DataFrame (e.g. 'temperature', 'bbTemperature', 'Puw').   |
| **out_directory**    | Directory to save the output DataFrames. Each row that is re-fit will be saved as a single-row CSV file.    |
| **fig_dpi**    | Figure DPI. Use this to scale the figures for your display, if needed.    |
| **start_data_ix**    | Index to start from, if picking up from previous work.    |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from citkid.res.interactive_sweep_fitter import resSweepFitter
%matplotlib inline
plt.ioff(); # These lines are necessary for the plots to display correctly

In [ ]:
def get_res_data(data_ix):
    """
    Function that loads data for the interactive sweep fitter. 
    
    Parameters:
    data_ix (int): data index, corresponding to the sweep of a single resonator.

    Returns:
    res_ix (int): resonator index.
    x (array-like (N,)): values (numeric) are the sweep parameter. N is
        the number of sweep points.
    y (array-like (N,)): values (numeric) are the resonance fit
        parameter for each dataset, corresponding to the values in x.
        For example, one may set x to temperature and y to fr.
    ffs (array-like, (N, M)): values (array-like) are arrays of
        frequencies for the fine S21 sweeps. M is the length of the S21
        sweep. Can be jagged.
    zfs (array-like, (N, M)): values (array-like) are arrays of
        complex S21 for the fine S21 sweeps. Can be jagged.
    fgs (array-like, (N, K)): values (array-like) are arrays of
        frequencies for the gain S21 sweeps. K is the length of the S21
        sweep. Can be jagged.
    zgs (array-like, (N, K)): values (array-like) are arrays of
        complex S21 for the gain S21 sweeps. Can be jagged.
    fres_alls (array-like, (N, L)): values (array-like) are arrays of
        resonant frequencies consisting of all resonant frequencies in
        the array, for gain sweep removal. L is the number of resonances
        in the array.
    qres_alls (array-like, (N, L)): values (array-like) are arrays of
        quality-factor-like parameters for removing resonances from the
        gain sweep, corresponding to the resonant frequencies in
        fres_alls.
    """
    # make sure the units of x are consistent with the 'x_unit' below
    
    # Be mindful of the performance of this function - when possible, don't 
    # reload from disk more than necessary 

In [ ]:
# Blackbody temperature
args = {'get_res_data': get_res_data,
        'resfit_param': 'fr',
        'x_name'      : 'BB temperature',
        'x_unit'      : 'K',
        'x_df_name'   : 'bbTemperature'
       }
# Bath temperature
args = {'get_res_data': get_res_data,
        'resfit_param': 'fr',
        'x_name'      : 'Temperature',
        'x_unit'      : 'mK', 
        'x_df_name'   : 'temperature'
       }
# Microwave power - you may want to do this in dB instead
args = {'get_res_data': get_res_data,
        'resfit_param': 'a',
        'x_name'      : r'$P_{\mu W}$',
        'x_unit'      : r'$\mu$W',
        'x_df_name'   : 'powerMicrowave' # normally, 'power' is in dB
       }
# nres and out_directory
args['nres'] = 
args['out_directory'] = 

rsf = resSweepFitter(**args)

In [ ]:
rsf.run_fitter()
# I recommend moving the output of this cell to a new window while working.

# get_res_data example
Here is an example of get_res_data for a blackbody sweep that could be copied if your data is stored in the standard format.

In [ ]:
directory = ''
# load data
data = pd.read_csv(directory + 'fitdata_noise_concat.csv')
data = data.sort_values(['resonatorIndex', 'bbTemperature'])
# Load the raw data only once per sweep index
row = data.iloc[0]
d0 = data[data.dataIndex == 0] # Just the first resonator
fres_alls, qres_alls = [], []
ffines, zfines, fgains, zgains = [], [], [], []
for index, row in d0.iterrows():
    fres_alls.append(np.load(row.dataDirectory + f'fres_all_{row.fileSuffix}.npy'))
    qres_alls.append(np.load(row.dataDirectory + f'qres_all_{row.fileSuffix}.npy'))
    f, i, q = np.load(row.dataDirectory + f's21_fine_{row.fileSuffix}.npy')
    ffines.append(f)
    zfines.append(i + 1j * q)
    f, i, q = np.load(row.dataDirectory + f's21_gain_{row.fileSuffix}.npy')
    fgains.append(f)
    zgains.append(i + 1j * q)

In [ ]:
resonator_indices = np.array(data.resonatorIndex.unique(), dtype = int)
resonator_indices = resonator_indices[resonator_indices >= 0]
nres = len(resonator_indices)
def get_res_data(data_ix):
    ri = int(resonator_indices[data_ix])
    d0 = data[data.resonatorIndex == ri].reset_index(drop = True)
    d0 = d0.sort_values('bbTemperature')
    ffs, zfs, fgs, zgs = [], [], [], []
    for ix, row in d0.iterrows(): 
        ffs.append(ffines[ix][row.dataIndex]) 
        zfs.append(zfines[ix][row.dataIndex])
        fgs.append(fgains[ix][row.dataIndex]) 
        zgs.append(zgains[ix][row.dataIndex])
    x_data = np.array(d0.bbTemperature)
    y_data = np.array(d0.iq_fr)
    return ri, x_data, y_data, ffs, zfs, fgs, zgs, fres_alls, qres_alls